# 03 — Gold: dim_date

| Property | Value |
|----------|-------|
| **Gold Table** | `dim_date` |
| **Grain** | One row per calendar date |
| **Source** | Generated — no silver source |
| **PK** | `DateKey` (int, YYYYMMDD) |
| **Rows** | ~13,150 |
| **Range** | 2000-01-01 to 2035-12-31 |

**Australian Fiscal Year**: July 1 → June 30. July 2025 = FY2026.

In [ ]:
# ============================================================
# Cell 1: Setup & Config
# ============================================================
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import date, timedelta

LAKEHOUSE = "The_Global_Loom"
TABLE = "dim_date"

DATE_START = date(2000, 1, 1)
DATE_END = date(2035, 12, 31)

print(f"✅ Config: {LAKEHOUSE}.{TABLE}")
print(f"   Range: {DATE_START} to {DATE_END}")

## Cell 2: Generate date spine

Create a list of every date in the range, then convert to a Spark DataFrame.

In [ ]:
# ============================================================
# Cell 2: Generate date spine
# ============================================================
days = (DATE_END - DATE_START).days + 1
date_list = [DATE_START + timedelta(days=i) for i in range(days)]

df_dates = spark.createDataFrame(
    [(d,) for d in date_list],
    schema=StructType([StructField("FullDate", DateType(), False)])
)

print(f"📅 Generated {df_dates.count():,} dates")
print(f"   From {DATE_START} to {DATE_END}")

## Cell 3: Transform — derive all date attributes

- **DateKey**: YYYYMMDD integer (e.g. 20260313)
- **Calendar fields**: Year, Quarter, Month, Day, DayOfWeek, MonthName, DayName
- **Australian Fiscal Year**: FY starts July 1. July 2025 = FY2026.
- **Fiscal Quarter**: FQ1 = Jul–Sep, FQ2 = Oct–Dec, FQ3 = Jan–Mar, FQ4 = Apr–Jun
- **Fiscal Month**: 1 = July, 2 = Aug, ..., 12 = June

In [ ]:
# ============================================================
# Cell 3: Transform — derive all date attributes
# ============================================================
df_dim_date = (
    df_dates
    # --- DateKey (PK) ---
    .withColumn("DateKey",
        (F.year("FullDate") * 10000 +
         F.month("FullDate") * 100 +
         F.dayofmonth("FullDate")).cast("int")
    )

    # --- Calendar fields ---
    .withColumn("Year", F.year("FullDate").cast("int"))
    .withColumn("Quarter", F.quarter("FullDate").cast("int"))
    .withColumn("Month", F.month("FullDate").cast("int"))
    .withColumn("Day", F.dayofmonth("FullDate").cast("int"))
    .withColumn("DayOfWeek", F.dayofweek("FullDate").cast("int"))  # 1=Sun in Spark
    .withColumn("MonthName", F.date_format("FullDate", "MMMM"))
    .withColumn("MonthNameShort", F.date_format("FullDate", "MMM"))
    .withColumn("DayName", F.date_format("FullDate", "EEEE"))
    .withColumn("YearMonth",
        (F.year("FullDate") * 100 + F.month("FullDate")).cast("int")
    )
    .withColumn("YearMonthLabel",
        F.date_format("FullDate", "yyyy-MM")
    )
    .withColumn("QuarterLabel",
        F.concat(F.lit("Q"), F.quarter("FullDate").cast("string"))
    )

    # --- Australian Fiscal Year (Jul-Jun) ---
    # If month >= 7 → FY = Year + 1, else FY = Year
    .withColumn("FiscalYear",
        F.when(F.month("FullDate") >= 7, F.year("FullDate") + 1)
         .otherwise(F.year("FullDate"))
         .cast("int")
    )
    .withColumn("FiscalYearLabel",
        F.concat(F.lit("FY"), 
                 F.when(F.month("FullDate") >= 7, F.year("FullDate") + 1)
                  .otherwise(F.year("FullDate"))
                  .cast("string"))
    )

    # --- Fiscal Quarter ---
    # FQ1=Jul-Sep (M7-9), FQ2=Oct-Dec (M10-12), FQ3=Jan-Mar (M1-3), FQ4=Apr-Jun (M4-6)
    .withColumn("FiscalQuarter",
        F.when(F.month("FullDate").between(7, 9), 1)
         .when(F.month("FullDate").between(10, 12), 2)
         .when(F.month("FullDate").between(1, 3), 3)
         .otherwise(4)
         .cast("int")
    )
    .withColumn("FiscalQuarterLabel",
        F.concat(
            F.lit("FY"),
            F.when(F.month("FullDate") >= 7, F.year("FullDate") + 1)
             .otherwise(F.year("FullDate")).cast("string"),
            F.lit(" Q"),
            F.when(F.month("FullDate").between(7, 9), F.lit("1"))
             .when(F.month("FullDate").between(10, 12), F.lit("2"))
             .when(F.month("FullDate").between(1, 3), F.lit("3"))
             .otherwise(F.lit("4"))
        )
    )

    # --- Fiscal Month (1=Jul, 2=Aug, ..., 12=Jun) ---
    .withColumn("FiscalMonth",
        F.when(F.month("FullDate") >= 7, F.month("FullDate") - 6)
         .otherwise(F.month("FullDate") + 6)
         .cast("int")
    )

    # --- Flags ---
    .withColumn("IsWeekend",
        F.dayofweek("FullDate").isin(1, 7)  # 1=Sun, 7=Sat in Spark
    )
    .withColumn("IsWeekday",
        ~F.dayofweek("FullDate").isin(1, 7)
    )
)

print(f"✅ Transformed: {df_dim_date.count():,} rows")
df_dim_date.printSchema()

## Cell 4: Final select with explicit column order

In [ ]:
# ============================================================
# Cell 4: Final select
# ============================================================
df_final = df_dim_date.select(
    F.col("DateKey").cast("int"),
    F.col("FullDate").cast("date"),
    F.col("Year").cast("int"),
    F.col("Quarter").cast("int"),
    F.col("QuarterLabel").cast("string"),
    F.col("Month").cast("int"),
    F.col("MonthName").cast("string"),
    F.col("MonthNameShort").cast("string"),
    F.col("Day").cast("int"),
    F.col("DayOfWeek").cast("int"),
    F.col("DayName").cast("string"),
    F.col("YearMonth").cast("int"),
    F.col("YearMonthLabel").cast("string"),
    F.col("FiscalYear").cast("int"),
    F.col("FiscalYearLabel").cast("string"),
    F.col("FiscalQuarter").cast("int"),
    F.col("FiscalQuarterLabel").cast("string"),
    F.col("FiscalMonth").cast("int"),
    F.col("IsWeekend").cast("boolean"),
    F.col("IsWeekday").cast("boolean")
)

df_final.show(5)
print(f"\n✅ Final schema: {len(df_final.columns)} columns")

## Cell 5: Data quality checks

In [ ]:
# ============================================================
# Cell 5: Data quality checks
# ============================================================
total = df_final.count()
dupes = total - df_final.select("DateKey").distinct().count()
nulls = df_final.filter(F.col("DateKey").isNull()).count()

# Spot-check fiscal year logic
spot = df_final.filter(F.col("FullDate") == "2025-07-01").collect()[0]
assert spot["FiscalYear"] == 2026, f"FY wrong: expected 2026, got {spot['FiscalYear']}"
assert spot["FiscalQuarter"] == 1, f"FQ wrong: expected 1, got {spot['FiscalQuarter']}"
assert spot["FiscalMonth"] == 1, f"FM wrong: expected 1, got {spot['FiscalMonth']}"

spot2 = df_final.filter(F.col("FullDate") == "2025-06-30").collect()[0]
assert spot2["FiscalYear"] == 2025, f"FY wrong: expected 2025, got {spot2['FiscalYear']}"
assert spot2["FiscalQuarter"] == 4, f"FQ wrong: expected 4, got {spot2['FiscalQuarter']}"
assert spot2["FiscalMonth"] == 12, f"FM wrong: expected 12, got {spot2['FiscalMonth']}"

print(f"✅ DQ Checks Passed")
print(f"   Total rows:     {total:,}")
print(f"   Duplicate keys: {dupes}")
print(f"   Null keys:      {nulls}")
print(f"   FY logic:       2025-07-01 → FY2026 Q1 ✅")
print(f"                   2025-06-30 → FY2025 Q4 ✅")

## Cell 6: Write to gold lakehouse

In [ ]:
# ============================================================
# Cell 6: Write to gold lakehouse
# ============================================================
df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABLE)

print(f"✅ Written: {TABLE}")
print(f"   Rows: {spark.table(TABLE).count():,}")